# Pandas: DataFrames for Tabular Data

Tuesday we met NumPy: fast, n-dimensional arrays with a single dtype. Real data is messier. A sales table mixes countries (text), dates, order IDs (integers), and prices (floats), and every column has a name you care about. NumPy alone is not the right shape for that.

Pandas is. A pandas `DataFrame` is a table with named columns that can hold different types. Each column is a `Series`, which is really a NumPy array with labels bolted on. Pandas is built on top of NumPy, so everything you learned about shape, axis, and aggregation carries forward.

This is the last notebook. The goal is a working tour, not mastery. You'll see the moves you already know (select a column, filter rows, summarize, group, sort, write to disk) expressed the pandas way, and you'll see the `sqlite3` + comprehension + `Counter` pipeline from 14a2 compress to a couple of method calls.

## Import

In [ ]:
import numpy as np
import pandas as pd

`pd` is the universal alias for pandas, the same way `np` is for NumPy. Use it.

## Two Data Structures

Pandas has two core objects: the `Series` (one column) and the `DataFrame` (a whole table).

### Series

A `Series` is a 1D labeled array. You can build one from a list.

In [ ]:
prices = pd.Series([19.99, 4.50, 12.00, 7.25])
prices

The output has three parts worth naming:

- The *values* on the right - a NumPy array under the hood.
- The *index* on the left - here, default integer positions `0..3`.
- The *dtype* - inferred from the values.

In [ ]:
print("values:", prices.values)
print("type of values:", type(prices.values))
print("index:", prices.index)
print("dtype:", prices.dtype)

You can give the index meaningful labels.

In [ ]:
prices = pd.Series(
    [19.99, 4.50, 12.00, 7.25],
    index=["widget", "sticker", "mug", "pen"],
)
prices

Now the Series works a lot like a dictionary - look up a value by its label.

In [ ]:
prices["mug"]

In fact, you can build a Series directly from a dict:

In [ ]:
pd.Series({"widget": 19.99, "sticker": 4.50, "mug": 12.00, "pen": 7.25})

### DataFrame

A `DataFrame` is a collection of Series that share a common row index - think of it as a dict of columns.

In [ ]:
df = pd.DataFrame({
    "product": ["widget", "sticker", "mug", "pen"],
    "price":   [19.99, 4.50, 12.00, 7.25],
    "stock":   [120, 5000, 40, 800],
})
df

Each column is a Series:

In [ ]:
df["price"]

In [ ]:
type(df["price"])

The mental model for the rest of the notebook:

- A DataFrame is a table.
- Each column is a `Series`, which is a NumPy array plus labels.
- Rows are observations, columns are attributes.
- Most analysis works column-by-column, because that is how questions are usually posed ("what is the average price?", "how many items are out of stock?").

### Where This Fits

You've now seen four ways to hold data:

- `list` - mixed types, 1D, integer position only, slow for numeric work
- `ndarray` - one type, N-D, integer position only, fast
- `Series` - one type, 1D, labeled index, fast (wraps an ndarray)
- `DataFrame` - mixed types by column, 2D, row index + column names, fast per column

Each one adds something on top of the previous. A `Series` is an `ndarray` plus labels. A `DataFrame` is a collection of `Series` sharing a row index. Pandas isn't a replacement for NumPy; it's NumPy with the structure tabular data actually has.

## Loading and Inspecting

Usually your data lives in a file, not a Python literal. `pd.read_csv` reads a CSV and returns a DataFrame. We'll use `5k-sales.csv` - the same data you saw in 14a2 as a SQLite table.

In [ ]:
sales = pd.read_csv("data/5k-sales.csv")

No loop, no `csv.DictReader`, no splitting on commas. One function call.

A handful of methods tell you what you just loaded.

In [ ]:
sales.head()

`.head()` shows the first 5 rows; pass an integer for a different count. `.tail()` shows the last rows.

In [ ]:
sales.shape

Shape is `(rows, columns)` - 5000 orders, 11 attributes each.

In [ ]:
sales.columns

In [ ]:
sales.dtypes

Numeric columns get `int64` or `float64`; text columns show up as `object`.

`.info()` combines several of these and adds non-null counts, which is the quickest way to spot missing data.

In [ ]:
sales.info()

And `.describe()` gives summary statistics for every numeric column in one shot.

In [ ]:
sales.describe()

This one method replaces a dozen lines of `sum()` / `mean()` / `min()` / `max()` loops.

## Selecting

Four selection patterns cover most of what you'll do.

### One column by name

Brackets with a string return a Series.

In [ ]:
sales["Country"].head()

### Multiple columns by name

Brackets with a *list* of strings return a DataFrame.

In [ ]:
sales[["Country", "Item_Type", "Units_Sold"]].head()

The double brackets trip people up. Read it as "bracket access, argument is a list": `sales[ ["Country", "Item_Type"] ]`.

### Rows by position with `.iloc`

Like NumPy indexing: `.iloc[row]` or `.iloc[row, col]`.

In [ ]:
sales.iloc[0]

The row prints *vertically* - the column names run down the left, the values down the right. That's because `.iloc[0]` returns a single row as a `Series`, and a Series always displays as a column of (label, value) pairs. The original columns become the Series index. Same data, different orientation for reading.

In [ ]:
sales.iloc[:3]

### Rows by condition (boolean filtering)

This is the big one - the pandas equivalent of SQL's `WHERE`.

A comparison on a column produces a Series of `True`/`False` values:

In [ ]:
big_orders = sales["Units_Sold"] > 8000
big_orders.head()

Pass that boolean Series back into brackets and you get only the rows where it is `True`:

In [ ]:
sales[sales["Units_Sold"] > 8000].head()

Compare the SQL you would have written:

```sql
SELECT * FROM sales WHERE Units_Sold > 8000 LIMIT 5;
```

Same idea, same result, different syntax. Combine conditions with `&` (and) and `|` (or) - each side needs its own parentheses:

In [ ]:
asia_bulk = sales[(sales["Region"] == "Asia") & (sales["Units_Sold"] > 8000)]
asia_bulk.head()

Rule of thumb:

- `df["col"]` or `df[["col1", "col2"]]` - columns by name
- `df.iloc[...]` - rows (and cells) by position
- `df[boolean_series]` - rows by condition

There is also `.loc[]` for label-based access. With the default integer index it overlaps with `.iloc[]`; it is useful when your index holds something meaningful, like dates or IDs.

## Column Arithmetic

Columns are Series, which are NumPy arrays with labels, so arithmetic is vectorized - no loops.

In [ ]:
revenue = sales["Units_Sold"] * sales["Unit_Price"]
revenue.head()

Assign the result back and you have a new column:

In [ ]:
sales["Revenue"] = sales["Units_Sold"] * sales["Unit_Price"]
sales["Profit"] = sales["Units_Sold"] * (sales["Unit_Price"] - sales["Unit_Cost"])
sales[["Units_Sold", "Unit_Price", "Unit_Cost", "Revenue", "Profit"]].head()

Aggregations on a Series collapse it to a single number, same as NumPy:

In [ ]:
print("total revenue:", sales["Revenue"].sum())
print("mean profit: ", sales["Profit"].mean())
print("max order:   ", sales["Units_Sold"].max())

## Plotting

Pandas delegates plotting to matplotlib through the `.plot` accessor on Series and DataFrames. It is not a full visualization library, but for quick looks during analysis it is hard to beat.

A histogram of a single column shows its distribution.

In [ ]:
sales["Units_Sold"].plot.hist(bins=30, title="Distribution of order sizes");

Orders are roughly uniform between 1 and 10,000 - the data is synthetic, which is why it looks that flat. Real sales data would pile up at smaller values.

A scatter plot shows the relationship between two columns.

In [ ]:
sales.plot.scatter(x="Unit_Price", y="Unit_Cost", alpha=0.3, title="Cost vs. Price");

Unit cost tracks unit price almost perfectly - that is, margins look proportional across items. `alpha=0.3` makes points semi-transparent so we can see clusters instead of a solid wall.

There is a whole world of visualization here (seaborn, matplotlib directly). You've now seen enough to explore on your own: `df.plot.bar(...)`, `df.plot.line(...)`, `df.plot.box(...)`, etc. all follow the same pattern.

## Group By

In SQL, you wrote

```sql
SELECT Region, SUM(Units_Sold) AS total
FROM sales
GROUP BY Region;
```

to get one row of summary per region. Pandas has the same operation, spelled `.groupby()`.

In [ ]:
sales.groupby("Region")["Units_Sold"].sum()

Read it as a pipeline: *group the rows by region, pick the `Units_Sold` column, sum within each group.* The result is a Series whose index is the group key.

Swap the aggregation for a different question:

In [ ]:
sales.groupby("Region")["Revenue"].mean().round(2)

Group by more than one column for a multi-level summary:

In [ ]:
sales.groupby(["Region", "Sales_Channel"])["Revenue"].sum().round(2)

And use `.agg({...})` when you want several aggregations at once - one dict entry per column:

In [ ]:
sales.groupby("Region").agg({
    "Units_Sold": "sum",
    "Unit_Price": "mean",
    "Revenue": "sum",
}).round(2)

Groupby is the single most useful pandas verb. Any question that starts with "for each...".

## Sort and Export

`.sort_values()` sorts a DataFrame by one or more columns.

In [ ]:
sales.sort_values("Revenue", ascending=False).head()

Sort by several columns by passing a list. `ascending` can take a matching list of booleans when you want different directions per column:

In [ ]:
sales.sort_values(
    ["Region", "Revenue"],
    ascending=[True, False],
).head()

Note that `sort_values` returns a *new* DataFrame - the original is untouched. This is a pandas idiom: most methods return new objects rather than modifying in place. If you want to keep the sorted version, assign it.

Writing results back to a CSV is `to_csv`:

In [ ]:
top_orders = sales.sort_values("Revenue", ascending=False).head(10)
top_orders.to_csv("data/top_orders.csv", index=False)

`index=False` omits the row-number index column - you almost always want this when exporting. Compare to the `csv.DictWriter` loop from 14a2; the DataFrame version is two lines.

## From SQL to DataFrame

You already know how to query SQLite in Python: open a connection, get a cursor, run `execute`, loop over `fetchall()`. Pandas short-circuits all of that with `pd.read_sql_query`.

In [ ]:
import sqlite3

with sqlite3.connect("data/5k-sales.db") as con:
    asia = pd.read_sql_query(
        "SELECT Country, Item_Type, Units_Sold, Unit_Price "
        "FROM sales WHERE Region = ?",
        con,
        params=("Asia",),
    )

asia.head()

One function call. The SQL runs inside the database; the results land directly in a DataFrame, ready for groupby, sorting, plotting, whatever.

Think back to 14a2. To find the top 5 selling countries in Asia, you wrote something like this: set `con.row_factory = sqlite3.Row`, run a query, collect `row["Country"]` and `row["Units_Sold"]` with a comprehension, build a `Counter`, call `.most_common(5)`. That whole pipeline becomes:

In [ ]:
asia.groupby("Country")["Units_Sold"].sum().sort_values(ascending=False).head(5)

This is the reason data scientists reach for pandas. SQL does the heavy filtering and joining in the database; pandas handles the analytic work once the rows are in memory. The two tools are complements, not competitors.

## Outside the Notebook

Notebooks are great for exploration. For anything you want to run on a schedule, hand to a colleague, or launch from another program, you move the code to a `.py` file.

Here is what that looks like. The file `15b_sales_digest.py` sits next to this notebook and runs from the terminal:

```text
$ python 15b_sales_digest.py Asia

=== Sales Digest: Asia ===
Orders:        719
Total units:   3,620,036
Total revenue: $920,277,085.92

Top 5 countries by units sold:
  Myanmar                   199,967
  South Korea               189,211
  Taiwan                    170,134
  Uzbekistan                163,904
  Sri Lanka                 153,186

Top 5 item types by revenue:
  Household            $ 235,400,780.58
  Office Supplies      $ 148,658,218.80
  Cosmetics            $ 126,092,414.80
  Meat                 $ 101,357,806.83
  Baby Food            $  86,088,074.40

Saved 719 rows to digest_asia.csv
```

You can run it from a Jupyter cell too, by prefixing with `!` to send the line to the shell:

In [ ]:
!python 15b_sales_digest.py Asia

A few things worth noticing when you open the file:

- *Imports at the top.* Standard-library imports (`sqlite3`, `sys`, `pathlib`) come first, then a blank line, then third-party imports (`pandas`). This is a near-universal Python convention.
- *Small functions.* `load_region`, `top_countries`, `top_items`, `print_report` each do one thing. `main` orchestrates them. Same discipline we've used all semester.
- *`if __name__ == "__main__":`* at the bottom. This line is Python's way of asking "am I being run as a script, or imported as a module?" When you run `python 15b_sales_digest.py`, Python sets a special variable called `__name__` to the string `"__main__"` and the block runs. When another file writes `import sales_digest`, `__name__` gets set to `"sales_digest"` instead and the block is skipped - so you can import `top_countries` into another script without triggering a report. It is the standard entry-point pattern; you'll see it in almost every `.py` file you encounter.
- *Command-line arguments.* `sys.argv` is a list of the words typed on the command line. `sys.argv[0]` is the script name itself; `sys.argv[1]` is the first argument. Real programs use `argparse` for anything more complicated than one argument, but the basic idea is the same: the user types something, the script reads it.
- *Exit codes.* `main()` returns `0` for success and `1` for failure. `sys.exit(main())` passes that number back to the shell. Scripts you chain together rely on this - a non-zero exit tells the next step "stop, something broke."

Same Python you've been writing. Different container.

## What We Skipped

There is a lot to Pandas. Things we didn't cover but you'll encounter:

- *Merges* - `pd.merge(df1, df2, on="key")` is the pandas version of SQL `JOIN`.
- *Missing data* - `isna`, `fillna`, `dropna`, and why `NaN` propagates through arithmetic.
- *Reshaping* - `pivot`, `melt`, `stack`, `unstack` for turning long data wide or vice versa.
- *`apply`* - running a function down a column when vectorized operations aren't enough.
- *Dates* - `pd.to_datetime`, the `.dt` accessor for pulling parts out of dates.
- *Real plotting* - `matplotlib.pyplot` directly, or `seaborn` for statistical graphics.

The pandas docs at <https://pandas.pydata.org/docs/> are good. When you're trying to do something specific, searching "pandas `<verb>`" almost always lands on a clear answer.

## Wrap-up

That's the tour. You now have:

- Loops, functions, collections, and comprehensions (base Python).
- Reading, filtering, joining data with SQL.
- Connecting the two with `sqlite3`, plus file I/O and CSV.
- NumPy for fast numerical work.
- Pandas for analytical work on tabular data.

Reminders:

- *Project* due Friday (Apr 24).
- *Final Exam* Monday (Apr 27), 1:30-3:30, here in Mell 2510.

Remaining class time is for project and final questions.

## Problems

These are not required. Work them if you want extra practice before the final.

**★ 1. Build a DataFrame**

Build a DataFrame called `inventory` from this data, then print its shape and dtypes:

- products: widget, sticker, mug, pen
- prices: 19.99, 4.50, 12.00, 7.25
- stock: 120, 5000, 40, 800

In [ ]:
# your code here

#### Solution

In [ ]:
inventory = pd.DataFrame({
    "product": ["widget", "sticker", "mug", "pen"],
    "price":   [19.99, 4.50, 12.00, 7.25],
    "stock":   [120, 5000, 40, 800],
})
print("shape:", inventory.shape)
print("dtypes:\n", inventory.dtypes, sep="")

**★ 2. Filter and Summarize**

Using the `sales` DataFrame, how many *offline* orders were there in Europe? Return a single integer.

In [ ]:
# your code here

#### Solution

In [ ]:
mask = (sales["Region"] == "Europe") & (sales["Sales_Channel"] == "Offline")
count = sales[mask].shape[0]
print(count)

Alternative - chain the filter into `.shape`:

In [ ]:
sales[(sales["Region"] == "Europe") & (sales["Sales_Channel"] == "Offline")].shape[0]

**★★ 3. Top Items by Revenue**

Return a Series of total revenue per `Item_Type`, sorted from highest to lowest. Show the top 5.

In [ ]:
# your code here

#### Solution

In [ ]:
(
    sales.groupby("Item_Type")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .round(2)
)

**★★ 4. SQL to DataFrame to Summary**

Using `pd.read_sql_query` against `5k-sales.db`, load only `Region`, `Units_Sold`, and `Unit_Price` columns, add a `Revenue` column, and return mean revenue per region sorted descending.

In [ ]:
# your code here

#### Solution

In [ ]:
with sqlite3.connect("data/5k-sales.db") as con:
    df = pd.read_sql_query(
        "SELECT Region, Units_Sold, Unit_Price FROM sales",
        con,
    )

df["Revenue"] = df["Units_Sold"] * df["Unit_Price"]
df.groupby("Region")["Revenue"].mean().sort_values(ascending=False).round(2)